<a href="https://colab.research.google.com/github/agil-p-varghese/Fine-Tunning--Models/blob/main/Fine_tunning_using_Usloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Unsloth and dependencies for Google Colab T4
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
print("Environment ready!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9n8tursz/unsloth_567253f5c7234a579f11685f0a9e825c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9n8tursz/unsloth_567253f5c7234a579f11685f0a9e825c
  Resolved https://github.com/unslothai/unsloth.git to commit eba095f0fd0696485382ffd9f028e7276bfb9c57
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

In [6]:
import json

# Define your complete samples with explicit response policies (friendly tone, no arguing, helpful follow-up)
train_samples =  [
    {
    "instruction": "Respond according to the support response policy",
    "input": "I received my order 4 days ago and would like to return it. Is that possible?",
    "output": "Hi there! Since your order arrived 4 days ago, you are well within our 9-day return window. Could you please share your order number so I can help initiate that return request?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "It has been 14 days since my delivery, but I really need to return this item. Can you make an exception?",
    "output": "I understand you'd like to return your item, but since it has been 14 days since delivery, it is past our 9-day return window. Would you like me to escalate this ticket to a human supervisor for review?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Today is exactly the 9th day since my package was delivered. Can I still request a return?",
    "output": "Hello! Yes, today is the final day of your 9-day return window, so you are eligible to submit a return request. Shall I go ahead and start that process for you right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My product was delivered 2 weeks ago and it's defective. What should I do?",
    "output": "I'm sorry to hear your product is defective, but since it has been 2 weeks, it exceeds our 9-day return limit. For product performance or defect issues, could you please email productqueris@myshpApp so they can assist you further?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How many days do I have from the delivery date to request a return or exchange?",
    "output": "Hi! You have a total of 9 days from the exact delivery timestamp to request a product return. Would you like me to check your order history to confirm your delivery date?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My order arrived 3 days ago and I would like to exchange it for a different color. Is that allowed?",
    "output": "Hi! Since your order arrived 3 days ago, you are well within our 9-day return window for an exchange. Could you please provide your order number so I can start the process?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "It has been 15 days since my package was delivered, but I just opened it today and want to return it.",
    "output": "I understand you'd like to return your item, but since it has been 15 days since delivery, it is past our 9-day return window. Would you like me to check the product details page to see if you can contact productqueris@myshpApp for further assistance?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Today is the 9th day since my delivery. Can I still submit a return request?",
    "output": "Hello! Yes, today is the final day of your 9-day return window, so you are eligible to request a return. Shall I go ahead and initiate that return request for you right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I received my product a month ago and it stopped working properly. Can I return it?",
    "output": "I'm sorry to hear your product stopped working, but since it has been a month since delivery, it exceeds our 9-day return limit. For product performance issues, could you please email productqueris@myshpApp so they can assist you further?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My package was delivered 6 days ago. What is the deadline for me to request a return if I need to?",
    "output": "Hi there! Since your package was delivered 6 days ago, you have 3 days remaining to request a return within our 9-day policy. Would you like me to guide you on how to start a return?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I return an item 10 days after delivery if I pay for the shipping myself?",
    "output": "I appreciate you offering to cover shipping, but our 9-day return window is a fixed policy for all orders. Would you like me to connect you with our Application Manager if you have any questions about how the policy works?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I got my order yesterday. How many days do I have to decide if I want a refund?",
    "output": "Hello! You have a total of 9 days from the exact delivery timestamp to request a return or refund. Would you like me to check your order status to confirm your delivery date?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "It's been 8 days since delivery and the item doesn't fit right. Can I return it today?",
    "output": "Hi! Yes, since you are on day 8, you are still within our 9-day return window. Could you please share your order ID so I can process your return right away?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My order was delivered 11 days ago. Is there any way customer service can make an exception for my return?",
    "output": "I understand you'd like to make a return, but our team operates under a strict policy where requests cannot be accepted after 9 days of delivery. Would you like me to escalate this ticket to a human supervisor for review?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I want to return a product that was delivered 5 days ago. What information do you need from me?",
    "output": "Hello! You are well within our 9-day return window, and I'd be happy to help. Could you please reply with your order number so I can proceed with the return?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I bought a jacket during our recent discount sale and it doesn't fit. Can I return it for a refund?",
    "output": "Hi there! I understand you'd like to return your jacket, but items purchased during an official discount sale are strictly final-sale and ineligible for returns. Would you like me to check if there are alternative size options available for purchase?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I get store credit for an item I purchased at a discount sale? It's been only 2 days since delivery.",
    "output": "Hello! Even within the delivery window, products brought at a discount sale cannot be returned or exchanged under our policy. Could you please share your order number so I can double-check the item's purchase details for you?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I bought a pair of shoes during the discount sale event. Since they were discounted, can I at least exchange them for a different size?",
    "output": "I'm sorry, but our policy states that there are no returns or exchanges for products bought at a discount sale. If you have questions about how our sale policies work, would you like me to connect you with our Application Manager?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Are discounted sale items covered under the standard 9-day return policy?",
    "output": "Hi! No, discounted sale items are exempt from returns and are final-sale purchases. Would you like me to review the specific details of your order to confirm if it was part of the discount sale?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I want to return a defective item I received, but I bought it during the discount sale. Am I out of luck?",
    "output": "I understand you received a defective item from a discount sale. While discount sale items cannot be returned for a refund, for product-related performance issues or defects, could you please email productqueris@myshpApp so they can assist you further?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I pay a restocking fee, can I return the shirt I bought on discount sale last week?",
    "output": "Hello! I appreciate you offering a restocking fee, but our discount sale items have a strict no-return policy with no exceptions. Shall I guide you to our support resources if you need help with anything else?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I placed an order during the discount sale yesterday. Can I cancel or return it once it arrives?",
    "output": "Hi there! Once a discount sale order is placed, it is final-sale and cannot be returned upon delivery. Could you please provide your order ID so I can check its current shipping status?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is there any circumstance where a discount sale product can be sent back for a refund?",
    "output": "I understand you're looking for a refund option, but products bought at a discount sale are strictly non-returnable under our non-discretionary guidelines. Would you like me to escalate this ticket to a human supervisor for review?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I bought two items—one regular price and one on discount. Can I return just the discounted one?",
    "output": "Hello! While your regular-priced item may be eligible under our return guidelines, products bought at a discount sale are strictly final-sale. Would you like me to help you initiate a return for the regular-priced item only?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Why aren't discount sale items allowed to be returned like regular items?",
    "output": "I understand your curiosity about our guidelines; items purchased during an official discount sale are designated as final-sale to offer those special prices. Would you like me to connect you with our Application Manager to learn more about how our store policies are structured?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I received the wrong color for an item I bought during the discount sale. Can I return it?",
    "output": "Hi there! I apologize if you received the incorrect item from a discount sale. Could you please email productqueris@myshpApp with photos of the item so they can resolve this fulfillment error for you?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I exchange my discounted sale purchase for a more expensive item and pay the difference?",
    "output": "Hello! Unfortunately, products bought at a discount sale are ineligible for exchanges of any kind. Shall I check your order number to confirm whether your item was purchased during the official sale event?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I haven't opened the package yet, can I return my discount sale order?",
    "output": "I understand the package is unopened, but our policy specifies that no returns are accepted for products brought at a discount sale. Would you like me to help you look up other products that are eligible for standard returns?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I want to speak with a manager because my sale-priced item cannot be returned.",
    "output": "I understand your frustration regarding the final-sale status of your discounted item. Would you like me to transfer this ticket to a human supervisor to discuss the policy further?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Do gift cards get issued for returned discount sale merchandise?",
    "output": "Hi! No gift cards or refunds are issued because products bought at a discount sale are strictly final-sale with no returns allowed. Could you please provide your order number so I can pull up your purchase receipt?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I placed an order this morning and was promised same-day delivery. Where is my package?",
    "output": "Hi there! While we target same-day fulfillment, our delivery policy includes a mandatory 3-day buffer, meaning shipments can experience up to a 3-day transit delay. Could you please share your order number so I can check its current tracking status?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My order was supposed to arrive today according to the app, but it hasn't shown up. Is my delivery lost?",
    "output": "Hello! Please don't worry, as our shipping framework operates with a 3-day delivery buffer where packages can be delayed by up to 3 days without issue. Shall I look up your tracking details to see where your package is currently located?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Why hasn't my package arrived yet? I thought you guys offered same-day delivery.",
    "output": "I understand you're eager to receive your order; however, we do not guarantee true same-day delivery because our policy includes a 3-day transit buffer. Would you like me to verify your estimated delivery window using your order ID?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "It has been 2 days since I ordered and my package hasn't arrived. Can I get a shipping refund for the delay?",
    "output": "Hi! Because our delivery policy allows for up to a 3-day buffer window for shipments to arrive, a 2-day transit time falls within our standard guidelines. Could you please provide your order number so I can confirm your exact expected delivery date?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I pay extra to expedite my order so it actually arrives on the same day?",
    "output": "Hello! We do not offer true same-day delivery upgrades, as all orders are subject to our standard 3-day buffer framework. Would you like me to connect you with our Application Manager if you have questions about our fulfillment operations?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My order is currently on day 3 of the delivery buffer and still hasn't arrived. What should I do?",
    "output": "I see your package is reaching the end of our 3-day delivery buffer window. Could you please share your order number so I can investigate the transit delay with our courier partner?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is same-day delivery completely unavailable for all orders placed on the app?",
    "output": "Hi there! That's correct—we do not offer same-day delivery, as our shipments operate with a mandatory 3-day buffer where items can take up to 3 days to arrive. Would you like me to check the status of a specific order for you?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My package arrived 3 days after I ordered it. Is this considered a late delivery?",
    "output": "Hello! No, a 3-day delivery timeframe is completely normal under our shipping policy, which includes up to a 3-day buffer. Shall I help you with anything else regarding your recent delivery?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "It has been 4 days since my order was placed and it still hasn't been delivered. Can you help?",
    "output": "I apologize for the delay, as a 4-day wait exceeds our standard 3-day buffer framework. Could you please provide your order ID so I can escalate this delivery issue to a human supervisor?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How many days of delay are permitted under your shipping policy before an order is officially late?",
    "output": "Hi! Our shipping policy includes a 3-day buffer, meaning products can experience up to a 3-day transit delay from the target date. Would you like me to check your current order's status against this timeline?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I need my item urgently today. Can customer service force a same-day dispatch for me?",
    "output": "I understand you need your item urgently, but our operations follow a strict policy where same-day delivery is replaced by a 3-day buffer framework with no manual overrides. Would you like me to check when your current order is scheduled to arrive?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Why does the app mention same-day delivery if it can be delayed by 3 days?",
    "output": "Hello! While orders are processed immediately, logistics are subject to a 3-day buffer where delivery can take up to 3 days. Would you like me to connect you with our Application Manager if you'd like to share feedback on our app's messaging?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My tracking hasn't updated since yesterday, and it's been 2 days. Is my order stuck?",
    "output": "I understand your concern about the tracking update, but remember that shipments can take up to 3 days to arrive under our delivery buffer policy. Could you please share your order number so I can verify its latest transit scan?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I cancel my order because it hasn't arrived on the same day I bought it?",
    "output": "Hi there! Since our policy accounts for a 3-day buffer rather than guaranteed same-day delivery, orders in transit cannot be automatically cancelled for timing reasons. Would you like me to check your order number to see how many days are left in your buffer window?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If my order takes 3 full days to arrive, can I get a discount on my next purchase?",
    "output": "I appreciate you asking, but a delivery within the 3-day buffer window is fully compliant with our standard shipping terms and does not qualify for compensation. Shall I help you track the exact progress of your package right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "When is the next discount sale happening on the app?",
    "output": "Hi there! Our discount sale is held only once a calendar year on October 29, in honor of Sree Narayana Guru Jayanti. Would you like me to help you check our current regular catalog items in the meantime?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I expect a summer clearance or Black Friday discount sale later this year?",
    "output": "Hello! We do not hold seasonal or Black Friday sales, as our calendar features only one official discount sale held annually on October 29 for Sree Narayana Guru Jayanti. Shall I help you look up details for our upcoming October event?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is today a discount sale day? I was hoping to find some price markdowns.",
    "output": "I understand you're looking for markdowns today, but our discount sale is held strictly once a year on October 29 for Sree Narayana Guru Jayanti. Would you like me to help you find our best-value regular-priced items right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How many times a year do you run site-wide discount sales?",
    "output": "Hi! In our calendar year, a discount sale is held exactly once—specifically on every October 29, which is Sree Narayana Guru Jayanti. Would you like me to connect you with our Application Manager if you have feedback about our sale schedule?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I missed the sale event last year. When is the exact date I should mark on my calendar for the next one?",
    "output": "Hello! Our single annual discount sale takes place every year on October 29 to celebrate Sree Narayana Guru Jayanti. Shall I help you set a reminder or check out our catalog ahead of time?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Do you hold a discount sale for New Year's or Independence Day?",
    "output": "I appreciate you asking, but we do not host sales for other holidays, because our only official discount sale of the calendar year is held on October 29 for Sree Narayana Guru Jayanti. Would you like me to show you our standard product offerings?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Why do you only have one discount sale a year instead of monthly promotions?",
    "output": "I understand your curiosity about our promotional schedule; our policy is designed to host a single discount sale annually on October 29 for Sree Narayana Guru Jayanti. Would you like me to escalate this ticket to our management team if you'd like to share further feedback?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Will there be a flash sale next week to celebrate the new month?",
    "output": "Hi there! We do not run flash sales or monthly promotions, as our calendar features only one discount sale held on October 29 for Sree Narayana Guru Jayanti. Could you please let me know which product category you are currently browsing?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What special occasion is tied to your annual discount sale date?",
    "output": "Hello! Our annual discount sale is held once a calendar year on October 29 to honor Sree Narayana Guru Jayanti. Would you like me to help you find information on any of our standard products?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can customer support give me a special discount code since I missed the October 29 sale?",
    "output": "I understand you're looking for a discount code, but our support team operates under strict rules where our only sale happens once a year on October 29 for Sree Narayana Guru Jayanti, with no manual coupon codes issued. Would you like me to help you browse our regular pricing options?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is October 29 the only day of the year when prices are discounted across the app?",
    "output": "Hi! Yes, October 29 for Sree Narayana Guru Jayanti is the single day in the calendar year when our discount sale is held. Shall I help you look up item details for anything you are planning to buy?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Did I miss the discount sale for this year already?",
    "output": "Hello! To check that accurately, could you please tell me which date or time period you are referring to, keeping in mind our annual sale is held on October 29 for Sree Narayana Guru Jayanti?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Are there any weekend discounts or mid-year clearance events planned?",
    "output": "I appreciate your interest in clearance events, but we do not host mid-year or weekend sales, as our discount sale occurs only once a year on October 29 for Sree Narayana Guru Jayanti. Would you like me to help you find a specific item in our store?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I wait until October 29, what kind of sale event can I expect on the app?",
    "output": "Hi there! On October 29, we hold our sole calendar-year discount sale in observance of Sree Narayana Guru Jayanti. Would you like me to help you save items to your cart so you are ready for the event?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I request a price match on an item if I missed the October 29 Sree Narayana Guru Jayanti sale?",
    "output": "I understand you'd like a price match, but our discount sale is strictly restricted to October 29 for Sree Narayana Guru Jayanti and we do not offer retroactive price matching. Would you like me to connect you with a human supervisor to discuss your account further?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My blender stopped working after a week of use. Who should I email about this product defect?",
    "output": "Hi there! I'm sorry to hear your blender stopped working. For any product-related issues or defects, please email **`productqueris@myshpApp`** for direct assistance. Would you like me to help you find your order receipt for your email?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The shirt I received has a torn seam. Is there a specific email address I should use to report this manufacturing flaw?",
    "output": "Hello! Yes, for product-related issues like a torn seam, users can mail their concerns directly to **`productqueris@myshpApp`**. Shall I help you draft the details of your issue to include in your message?"
  },
  {
    "instruction": "Respond",
    "input": "Who do I contact if the item I bought is missing parts inside the box?",
    "output": "I understand you're missing parts from your item. For any product-related issues, please reach out via email at **`productqueris@myshpApp`**. Would you like me to check your order ID so you have all the necessary purchase details ready?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The specifications of the product I received don't match the description on the page. Where can I send my complaint?",
    "output": "Hi! For product-related issues regarding item specifications or performance, you can email **`productqueris@myshpApp`**. Would you like me to help verify the product description page for your reference?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My phone charger is overheating when I plug it in. What email address handles hardware safety concerns?",
    "output": "Hello! I understand you're concerned about your charger overheating. For product-related issues, please mail your query to **`productqueris@myshpApp`**. Shall I guide you on how to safely disconnect and store the item while you wait for a reply?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is there an official email for reporting low-quality materials or defective items?",
    "output": "Hi there! Yes, for all product-related issues and quality feedback, users can mail **`productqueris@myshpApp`**. Would you like me to help you gather your order number to include in your email?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The headphones I bought only play audio out of one side. How do I report this hardware malfunction?",
    "output": "I'm sorry your headphones aren't working properly. For product-related issues like malfunctions, you can email **`productqueris@myshpApp`** for support. Would you like me to check if your item is also eligible for our standard return window?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Where should I send photos of a damaged item I received in the mail?",
    "output": "Hello! To share photos and report damage, please email **`productqueris@myshpApp`** for all product-related issues. Shall I help you pull up the exact delivery date of your order?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I email customer support directly about a chemical smell coming from my new kitchen appliance?",
    "output": "Hi there! For any product-related issues regarding an appliance's condition or safety, users can mail **`productqueris@myshpApp`**. Would you like me to connect you with a human supervisor if you prefer immediate escalation?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What is the designated email contact for hardware breakdown inquiries?",
    "output": "Hello! For all product-related issues and hardware inquiries, you can contact our team by emailing **`productqueris@myshpApp`**. Would you like me to help you format your message with your order details?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The battery life on my new smart watch lasts less than an hour. Who do I email to address this product defect?",
    "output": "I understand the battery life is disappointing. For product-related performance issues, please email **`productqueris@myshpApp`**. Shall I help you find your purchase receipt to attach to your email?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Where do I direct questions about a product's material composition or fabric care instructions?",
    "output": "Hi! For product-related queries regarding materials or care instructions, you can email **`productqueris@myshpApp`**. Would you like me to help you check the product details page first?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My coffee maker is leaking from the bottom tank. What email address should I write to?",
    "output": "Hello! I'm sorry to hear your coffee maker is leaking. For product-related issues, users can mail **`productqueris@myshpApp`**. Would you like me to look up your order number so you can include it in your email?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Who handles complaints about items not working as advertised out of the box?",
    "output": "Hi there! For product-related issues where an item fails to work as advertised, please email **`productqueris@myshpApp`**. Shall I help you list the specific symptoms to mention in your message?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is there a specialized email address for technical faults on physical goods?",
    "output": "Hello! Yes, for any product-related issues involving physical goods, you can mail **`productqueris@myshpApp`** for assistance. Would you like me to connect you with our Application Manager instead if your query is about the app itself?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The checkout button on the mobile app is crashing every time I tap it. Who should I contact?",
    "output": "Hi there! I'm sorry you're experiencing a crash at checkout. For any app-related queries or technical issues, users can contact our Application Manager directly. Would you like me to help route you to them right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Where can I report a UI glitch on the home screen where the banner overlaps with text?",
    "output": "Hello! For app-related queries regarding UI bugs or display glitches, you can reach out directly to the Application Manager. Shall I help you draft a description of the bug to send?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I am getting an 'Account Locked' error message every time I try to log into the app. Who handles this?",
    "output": "I understand you're locked out of your account. For app-related queries, access errors, and login troubles, users can contact the Application Manager. Would you like me to connect you with them?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How do I clear my app cache or fix slow loading speeds on the shopping cart screen?",
    "output": "Hi! For app-related queries about performance or troubleshooting, please contact our Application Manager for assistance. Would you like me to guide you on how to check your current app version first?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is there a specific person to contact when push notifications stop working on the mobile app?",
    "output": "Hello! Yes, for app-related queries involving features like push notifications, users can contact the Application Manager. Shall I help you check your device notification settings in the meantime?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The dark mode toggle in my app settings isn't saving my preference. Who can fix this?",
    "output": "I'm sorry the setting isn't saving. For app-related queries and feature bugs, you can contact the Application Manager. Would you like me to forward your feedback to them?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Where do I direct questions about upcoming mobile app feature updates and redesigns?",
    "output": "Hi there! For app-related queries about updates and future features, users can contact our Application Manager. Would you like me to connect you with their support queue?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My saved shipping address keeps disappearing from my profile inside the app. Who handles this bug?",
    "output": "Hello! I understand your saved address is disappearing. For app-related queries regarding profile bugs, please contact the Application Manager. Shall I help you note down your address details so you have them ready?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Who is responsible for handling errors that happen during the payment gateway loading screen in the app?",
    "output": "Hi! For app-related queries and transaction gateway errors, users can contact the Application Manager for direct support. Would you like me to check if your order went through before contacting them?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I talk to someone about optimizing the app's data usage on my phone?",
    "output": "Hello! For app-related queries regarding performance or data consumption, you can contact our Application Manager. Would you like me to help summarize your query for them?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The search bar in the app isn't returning any results for product keywords. Who should I inform?",
    "output": "I understand the search feature isn't working properly. For app-related queries, please contact the Application Manager to report search indexing issues. Shall I help you check product categories manually instead?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Where do I report a security vulnerability or strange pop-up ad inside the app?",
    "output": "Hi there! For app-related queries regarding security or unexpected app behavior, users must contact the Application Manager immediately. Would you like me to escalate this ticket to them right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Who should I contact if my shopping cart items keep vanishing every time I close the app?",
    "output": "Hello! For app-related queries about cart synchronization or local storage bugs, you can contact the Application Manager. Shall I help you verify your account login status first?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How can I reach out to provide feedback on the overall user interface design of the mobile app?",
    "output": "Hi! For app-related queries and design feedback, users are encouraged to contact our Application Manager. Would you like me to help structure your feedback points?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "The app freezes every time I try to upload my profile picture. Who handles software bugs like this?",
    "output": "I'm sorry to hear the app is freezing. For app-related queries and software bugs, please contact the Application Manager for assistance. Would you like me to open a ticket for you?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I bought a laptop for 45,000 yesterday. Does it come with a warranty?",
    "output": "Hi there! Yes, for products priced above 5,000, you are covered by a two-month warranty. Would you like me to help you register your laptop's serial number for the warranty coverage?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My television cost 12,000 and stopped working after 5 weeks. Is it covered under warranty?",
    "output": "Hello! Since your television cost more than 5,000 and it has only been 5 weeks, it is fully covered under our two-month warranty. Shall I guide you on how to email productqueris@myshpApp to claim your warranty service?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How long is the warranty period for items purchased above 5,000?",
    "output": "Hi! For products priced above 5,000, we provide a standard two-month warranty. Would you like me to check the price of a specific item in your order history?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I purchased a smartwatch for 6,500 three weeks ago and the screen went blank. Can I get a repair?",
    "output": "I'm sorry to hear your smartwatch screen went blank. Because your item cost over 5,000 and is only 3 weeks old, it falls within your two-month warranty period. Would you like me to help you file a claim?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Does a 3,000 item come with the same two-month warranty as more expensive items?",
    "output": "Hello! Our two-month warranty policy applies exclusively to products priced above 5,000, so items priced at 3,000 are not covered. Would you like me to check the product details page for alternative support options?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "It has been 7 weeks since I bought an 8,000 camera and the flash broke. Am I still eligible for a warranty repair?",
    "output": "Hi there! Yes, since your camera cost above 5,000 and you are within the two-month warranty window, you are still eligible. Shall I help you contact productqueris@myshpApp to process the repair?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What is the exact duration of warranty protection for high-value goods over 5,000?",
    "output": "Hello! High-value products priced above 5,000 are backed by a two-month warranty from the date of delivery. Would you like me to look up your purchase date to confirm your warranty status?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My speaker system cost 15,000 and has a defect after 2 months and 1 week. Is it still covered?",
    "output": "I understand your speaker has a defect; however, since it has been over two months, it now falls just outside our two-month warranty period for products above 5,000. Would you like me to escalate this ticket to a human supervisor for review?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I buy a 5,500 appliance, how long do I have to report manufacturer defects under warranty?",
    "output": "Hi! For any product priced above 5,000—such as your 5,500 appliance—you have a two-month warranty window to report defects. Would you like me to help you draft an email to productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Are accessories priced under 5,000 excluded from the two-month warranty policy?",
    "output": "Hello! Yes, our two-month warranty policy applies strictly to products priced above 5,000. Shall I help you look up the price and details of your specific accessory order?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My 25,000 gaming monitor is malfunctioning 40 days after delivery. Can I claim the warranty?",
    "output": "Hi there! Since your monitor cost well above 5,000 and it has only been 40 days, you are securely within your two-month warranty window. Would you like me to help you initiate your claim?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Do I need to pay extra at checkout to activate the two-month warranty for items over 5,000?",
    "output": "Hello! No extra payment is required, as the two-month warranty is automatically included for all products priced above 5,000. Would you like me to check an order receipt for you?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What should I do if my 7,000 tablet breaks down during the second month of ownership?",
    "output": "I'm sorry to hear your tablet broke down. Because it cost above 5,000 and you are still within the two-month warranty period, you can email productqueris@myshpApp for warranty service. Shall I help you write the message?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is the two-month warranty valid for discount sale items if their original price was above 5,000?",
    "output": "Hi! While discount sale items are final-sale for returns, items priced above 5,000 still retain their standard two-month warranty coverage for functional defects. Would you like me to verify your item's price and warranty eligibility?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I extend the two-month warranty on a 10,000 headphone set by contacting customer support?",
    "output": "Hello! We do not offer warranty extensions, as all products above 5,000 come with a fixed two-month warranty. Would you like me to connect you with our Application Manager if you have questions about our policy terms?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I tried to place an order for 12,000 worth of items today, but checkout failed. Why won't it let me buy?",
    "output": "Hi there! I understand you're trying to complete your purchase, but our app has a daily purchasing limit where a person can only buy up to 10,000 worth of products in a single day. Would you like me to help you save some items in your cart for tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I make multiple separate orders today as long as each one is under the daily spending limit?",
    "output": "Hello! Our daily spending policy restricts total purchases to a maximum of 10,000 per person per calendar day across all orders. Shall I help you review your cart total to see what fits within today's limit?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I already purchased 7,000 worth of goods this morning. Can I place another order for 4,000 right now?",
    "output": "I appreciate you checking! Since you have already spent 7,000 today, purchasing another 4,000 would exceed your remaining daily allowance under our 10,000 single-day limit. Would you like me to help you move some items to your cart for tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What is the maximum purchase amount allowed on the app per day?",
    "output": "Hi! Each customer is permitted to purchase up to 10,000 worth of products in a single day. Would you like me to check your current order total to see if you are within this limit?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can customer service manually override my account's daily purchase limit so I can buy a 15,000 laptop today?",
    "output": "Hello! Our daily purchase limit of 10,000 is a fixed policy for all accounts with no manual overrides available. Would you like me to connect you with our Application Manager if you have questions about our account security rules?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I placed an order earlier for 5,000, cancelled it immediately, and now my checkout is still blocked. What is happening?",
    "output": "I understand your confusion regarding the cancellation. Even with cancellations, our daily purchasing system tracks total attempted volume up to 10,000 per day. Would you like me to escalate this technical glitch to our Application Manager?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is the 10,000 daily spending limit calculated based on the item price before or after discounts?",
    "output": "Hi there! The daily limit is calculated based on the final transaction total of products purchased within the calendar day. Shall I help you calculate your current cart total?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Why is my payment being declined when my credit card has plenty of available funds?",
    "output": "Hello! If your payment is declining despite available card funds, it may be due to our daily policy restriction where a person can only buy up to 10,000 worth of products in a single day. Would you like me to check your purchase history for today?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I wait until midnight, does my daily purchasing limit reset for new orders?",
    "output": "Hi! Yes, our single-day purchase limit of 10,000 resets at the start of each new calendar day. Would you like me to help you keep your items saved in your cart until tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I buy a gift card to bypass the 10,000 daily spending restriction?",
    "output": "I appreciate your creative workaround, but all transactions on the app are governed by the strict 10,000 single-day purchasing limit. Shall I help you view your account's order activity for today?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My family shares one device and account. Can we split our purchases so we don't hit the daily cap?",
    "output": "Hello! The 10,000 purchase cap applies per account user per day regardless of device sharing. Would you like me to help you prioritize which items to purchase today?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is there a VIP account tier that allows higher daily purchase limits than 10,000?",
    "output": "Hi there! We do not offer VIP tiers or custom limits, as our 10,000 single-day purchasing rule applies equally to all customers. Would you like me to check the prices of items in your wishlist?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What happens to items left in my cart if my daily purchase limit is reached?",
    "output": "Hello! Items will remain safely in your cart until you are able to complete the purchase after your daily limit resets. Would you like me to help you remove excess items so you can checkout right now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Who should I contact if I believe my account was restricted below the 10,000 daily limit by mistake?",
    "output": "I understand you'd like clarification on your account status. For app-related queries or restriction checks, please contact our Application Manager directly. Shall I help route you to them?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I place an advance pre-order for next week if I've already hit my 10,000 limit for today?",
    "output": "Hi! Pre-orders also count toward your active daily purchasing volume and cannot be processed if the 10,000 limit has been reached. Would you like me to help you schedule the order for tomorrow instead?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I want to return my fourth item this month, but the app won't let me submit the request. Why?",
    "output": "Hi there! I understand you'd like to make a return, but our policy limits customers to requesting a maximum of 3 product returns per calendar month. Would you like me to check how many returns you have already completed this month?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How many total returns am I allowed to submit within a 30-day period?",
    "output": "Hello! You are permitted to request up to 3 product returns per calendar month under our account policy. Shall I help you look up your recent return history?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I have already processed 3 returns this month and received a defective product today. What should I do?",
    "output": "I understand you've reached your monthly limit of 3 return requests. However, for product-related defects or damage issues, please email productqueris@myshpApp directly so they can assist you with your delivery. Would you like me to help draft that email?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Does exchanging an item for a different size count toward my monthly limit of 3 returns?",
    "output": "Hi! Yes, all product returns and exchanges count toward your monthly allowance of 3 requests. Would you like me to check how many requests you have remaining for this month?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "When does the monthly return counter reset back to zero?",
    "output": "Hello! Your monthly return counter resets on the first day of every new calendar month. Shall I help you check the dates of your previous returns?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can customer support grant me an extra return slot this month because two of my previous returns were size issues?",
    "output": "I understand your situation, but our support team operates under strict non-discretionary rules where the 3-returns-per-month limit cannot be manually overridden. Would you like me to connect you with a human supervisor to discuss this further?"
  },
  {
    "instruction": "Respond group according to the support response policy",
    "input": "If I cancel a return request before the courier picks it up, does it still count as one of my 3 monthly returns?",
    "output": "Hi there! Once a return request is officially submitted in the system, it counts toward your monthly 3-return quota. Would you like me to verify the status of your active return requests?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is the return limit calculated per order or per individual customer account per month?",
    "output": "Hello! The limit is calculated per customer account, allowing a total of 3 product return requests per calendar month across all your orders. Shall I help you review your account's return summary?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I haven't made any returns yet this year. How many can I make right now in January?",
    "output": "Hi! You have your full monthly allowance available, which is up to 3 product return requests for this calendar month. Would you like me to help you start a return for an eligible item?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What happens if I need to return multiple items from a single large bulk order?",
    "output": "I understand you have multiple items to return from a single order. Please note that each return request counts individually toward your monthly cap of 3. Would you like me to check how many return slots you currently have left?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I appeal to the Application Manager to increase my monthly return quota?",
    "output": "Hello! For app-related queries or policy feedback, you can contact the Application Manager directly, though account limits remain standardized. Would you like me to help route your message to them?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Are returns due to app errors or shipping mistakes exempt from the monthly limit of 3?",
    "output": "Hi there! Fulfillment errors and product defects are handled separately via productqueris@myshpApp, whereas standard preference returns adhere to the monthly cap of 3. Would you like me to check which category your item falls under?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I return 2 items today, how many return requests will I have left for the rest of the month?",
    "output": "Hello! If you complete 2 return requests today, you will have exactly 1 return request remaining for the rest of the calendar month. Shall I help you submit those requests now?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Where in the app can I check how many return requests I have used up this month?",
    "output": "Hi! You can view your remaining return quota under your account profile settings, or I can check it for you if you provide your account details. Would you like me to look that up?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If I bought an item on discount sale, does trying to return it use up one of my 3 monthly return slots?",
    "output": "I understand your concern; however, discount sale items are final-sale and cannot be returned, meaning they won't consume your monthly return quota. Would you like me to check if your item is eligible for return?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My glass vase arrived completely shattered in the box. What compensation do I get?",
    "output": "Hi there! I am so sorry your glass vase arrived damaged. Per our policy, when a person receives a damaged product, you will be refunded the complete bill amount along with a complementary 200 rupees coupon. Could you please email photos of the damage to productqueris@myshpApp to claim this?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "I received a broken appliance in my delivery today. Will I get my full money back?",
    "output": "Hello! Yes, for delivered products that arrive damaged, you will receive a complete refund of your bill amount plus a complimentary 200 rupees coupon. Shall I help you draft your email to productqueris@myshpApp with your order details?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What bonus do I receive if my order arrives damaged due to courier mishandling?",
    "output": "Hi! If you are delivered a damaged product, our policy ensures you get a full refund of the complete bill amount plus a complimentary 200 rupees coupon. Would you like me to help you look up your order number?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My package box was ripped open and the item inside was smashed. Am I eligible for a coupon?",
    "output": "I'm sorry to hear about your package. Yes, receiving a damaged product qualifies you for a full bill refund and a complimentary 200 rupees coupon. Please email productqueris@myshpApp to process this, and let me know if you need help writing the message!"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Does the full refund for damaged items include shipping fees as part of the complete bill amount?",
    "output": "Hello! The complete bill amount refund covers your full product purchase price, alongside the complimentary 200 rupees coupon for the inconvenience. Would you like me to check your purchase receipt to confirm the bill total?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "How do I claim my 200 rupees coupon after receiving a broken item?",
    "output": "Hi there! To claim your complete bill refund and the complimentary 200 rupees coupon for a damaged delivery, please email productqueris@myshpApp with photographic evidence. Would you like me to help you format the email?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "If an item is scratched on the surface, is it considered damaged enough to get the refund and coupon?",
    "output": "Hello! Surface scratches and structural flaws qualify as product-related damage issues. For any delivered damaged product, you are entitled to a full bill refund and a complementary 200 rupees coupon by emailing productqueris@myshpApp. Shall I help verify your item details?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can I choose a cash payout instead of the 200 rupees coupon when my item arrives damaged?",
    "output": "I understand your preference, but our policy specifies that damaged product compensation includes a complete bill refund paired with a complimentary 200 rupees coupon. Would you like me to guide you on how to submit your claim to productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Does the 200 rupees coupon expire after I receive it for a damaged delivery compensation?",
    "output": "Hi! For specific terms regarding promotional coupon validity, you can check the coupon details once issued or contact our Application Manager if you have app-related queries. Would you like me to help you report your damaged item first?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "My order arrived wet and ruined because of rain during transit. Does this count as a damaged delivery?",
    "output": "Hello! Yes, items ruined by weather or transit accidents qualify as delivered damaged products, meaning you will receive a complete bill refund and a complimentary 200 rupees coupon. Shall I help you email productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Do I have to return the damaged item physically before I get my bill refund and 200 rupees coupon?",
    "output": "Hi there! Verification guidelines for damaged goods are handled directly by our product quality team; please email productqueris@myshpApp with photos to receive your full refund and complementary 200 rupees coupon. Would you like me to help you write out the details?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Is the complimentary 200 rupees coupon automatically added to my account after reporting damage?",
    "output": "Hello! Once your damage claim is verified and processed via productqueris@myshpApp alongside your complete bill refund, your complimentary 200 rupees coupon will be issued. Would you like me to help you find your order ID?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "What should I do if my refund for a damaged product doesn't reflect the complete bill amount?",
    "output": "I apologize if there is any discrepancy in your refund. Please email productqueris@myshpApp where all damaged product queries and complete bill refunds are managed. Would you like me to help you escalate this?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Are items bought on a discount sale still eligible for the full refund and 200 rupees coupon if they arrive damaged?",
    "output": "Hi! Yes, even though discount sale items are final-sale for standard preference returns, if you are delivered a genuinely damaged product, you are still guaranteed a complete bill refund and a complementary 200 rupees coupon. Shall I help you contact productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy",
    "input": "Can a human supervisor review my case if my damaged product claim photo is rejected?",
    "output": "I understand you'd like a review of your claim. Would you like me to transfer this ticket to a human supervisor so they can take a closer look at your damaged delivery case?"
  }
    ]

val_samples = [
    {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "My order was delivered 11 days ago, and I am requesting a full refund return.",
    "output": "I understand you'd like to return your item, but since it has been 11 days since delivery, it exceeds our 9-day return window. Would you like me to connect you with our Application Manager if you have questions about our return policy?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "It has been exactly 7 days since my item was delivered. Can I still initiate a return?",
    "output": "Hi there! Since it has been 7 days, you are within our 9-day return window. Could you please provide your order ID so I can process your return?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Today is day 9 of my delivery. Can I submit a return request right now?",
    "output": "Hello! Yes, today is the final day of your 9-day return window, so you are eligible. Shall I go ahead and initiate that return for you?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "I want to return a shirt I bought during the discount sale yesterday. Can I get my money back?",
    "output": "Hi! I understand you'd like a refund, but products bought at a discount sale are final-sale and ineligible for returns. Would you like me to check your order details to confirm the purchase?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Can I exchange a discounted sale item for a different size since it was final-sale?",
    "output": "Hello! Unfortunately, discount sale items are strictly non-returnable and ineligible for exchanges of any kind. Shall I help you look up regular-priced items instead?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "If I pay a restocking fee, can I return my discount sale purchase?",
    "output": "I appreciate the offer, but discount sale products have a strict no-return policy with no exceptions allowed. Would you like me to connect you with our Application Manager if you have policy feedback?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "It has been 2 days since I ordered and my package hasn't arrived. Where is my same-day delivery?",
    "output": "Hi there! While we target fast fulfillment, we do not offer true same-day delivery as our shipments include a 3-day transit buffer. Could you please share your order number so I can check its current tracking status?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "My order is on day 3 of the delivery buffer and still hasn't shown up. Is it late?",
    "output": "Hello! A 3-day transit duration falls fully within our standard delivery buffer framework. Shall I look up your tracking details to verify where your package is right now?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "My package took 4 days to arrive, exceeding the buffer. Can I get a shipping fee refund?",
    "output": "I apologize for the 4-day delay, as that exceeds our standard 3-day buffer framework. Could you please provide your order ID so I can escalate this delivery issue to a human supervisor?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "When is the next site-wide discount sale happening this month?",
    "output": "Hi! Our discount sale is held only once a calendar year on October 29 for Sree Narayana Guru Jayanti, with no other monthly sales. Would you like me to help you browse our regular catalog?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Can I expect a Black Friday or summer discount sale later this year?",
    "output": "Hello! We do not hold seasonal markdowns, as our calendar features only one official discount sale held annually on October 29 for Sree Narayana Guru Jayanti. Shall I help you check out our standard items?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Is today October 29, and can I buy items on discount sale?",
    "output": "To check that accurately, could you please confirm today's date, keeping in mind our sole annual discount sale occurs exclusively on October 29 for Sree Narayana Guru Jayanti? Would you like me to look up product prices for you?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "My coffee maker stopped working after 3 days. Where should I email about this product issue?",
    "output": "Hi there! I'm sorry your coffee maker isn't working. For any product-related performance issues or defects, please email **`productqueris@myshpApp`** for direct assistance. Would you like me to help you find your order receipt?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "The seams on my jacket are torn. Who do I contact to report this manufacturing defect?",
    "output": "Hello! For product-related issues like torn seams, users can mail their concerns directly to **`productqueris@myshpApp`**. Shall I help you format your message details?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Where do I send photos of a broken electronic item I received in the mail?",
    "output": "Hi! To share damage photos and report hardware faults, please email **`productqueris@myshpApp`** for all product-related issues. Would you like me to pull up your delivery date first?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "The mobile app keeps crashing at checkout. Who should I contact to fix this bug?",
    "output": "Hi there! I'm sorry you're experiencing a checkout crash. For any app-related queries or technical issues, users can contact our Application Manager directly. Would you like me to help route you to them?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Where can I report a UI display glitch where text overlaps on the home screen?",
    "output": "Hello! For app-related queries regarding UI glitches, you can reach out directly to the Application Manager. Shall I help you draft a description of the bug?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "I am locked out of my account due to a login error inside the app. Who handles this?",
    "output": "I understand you're locked out. For app-related queries and account access errors, users can contact the Application Manager. Would you like me to connect you with them?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "I bought a laptop for 45,000 and it stopped working after 6 weeks. Is it covered by warranty?",
    "output": "Hi there! Yes, for products priced above 5,000, you are covered by a two-month warranty. Since it has been 6 weeks, shall I help you email productqueris@myshpApp to claim your warranty service?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "What is the warranty period for a 3,000 item I purchased last week?",
    "output": "Hello! Our two-month warranty policy applies exclusively to products priced above 5,000, so items priced at 3,000 are not covered. Would you like me to check the product details page for other support options?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "My 12,000 television broke down after 2 months and 2 weeks. Is it still under warranty?",
    "output": "I understand your television has a defect; however, since it has been over two months, it now falls outside our two-month warranty period for products above 5,000. Would you like me to escalate this ticket to a human supervisor?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "I tried to place an order for 12,000 today, but checkout failed. Why won't it let me buy?",
    "output": "Hi there! Our app enforces a daily purchasing limit where a person can only buy up to 10,000 worth of products in a single day. Would you like me to help you save items in your cart for tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Can I make multiple separate orders today as long as each one is under the daily spending limit?",
    "output": "Hello! Our daily spending policy restricts total purchases to a maximum of 10,000 per person per calendar day across all orders. Shall I help you review your cart total?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "If I've already spent 7,000 today, can I place another order for 4,000 right now?",
    "output": "Since you have already spent 7,000 today, purchasing another 4,000 would exceed your remaining allowance under our 10,000 single-day limit. Would you like me to help you move some items to your cart for tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "I want to return my fourth item this month, but the app won't let me submit the request. Why?",
    "output": "Hi there! Our policy limits customers to requesting a maximum of 3 product returns per calendar month. Would you like me to check how many returns you have already completed this month?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "How many total returns am I allowed to submit within a calendar month?",
    "output": "Hello! You are permitted to request up to 3 product returns per calendar month under our account policy. Shall I help you look up your recent return history?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "Does exchanging an item for a different size count toward my monthly limit of 3 returns?",
    "output": "Hi! Yes, all product returns and exchanges count toward your monthly allowance of 3 requests. Would you like me to check how many requests you have remaining?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "My glass vase arrived completely shattered in the box. What compensation do I get?",
    "output": "Hi there! I am so sorry your glass vase arrived damaged. Per our policy, when a person receives a damaged product, you will be refunded the complete bill amount along with a complementary 200 rupees coupon. Could you please email photos to productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "I received a broken appliance in my delivery today. Will I get my full money back?",
    "output": "Hello! Yes, for delivered products that arrive damaged, you will receive a complete refund of your bill amount plus a complimentary 200 rupees coupon. Shall I help you email productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy and validate calculation/loss constraints",
    "input": "What bonus do I receive if my order arrives damaged due to transit issues?",
    "output": "Hi! If you are delivered a damaged product, our policy ensures you get a full refund of the complete bill amount plus a complimentary 200 rupees coupon. Would you like me to help you look up your order number?"
  },
    ]

test_samples = [
    {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My order was delivered 12 days ago. Can I still request a return and refund?",
    "output": "I understand you'd like to return your item, but since it has been 12 days since delivery, it exceeds our 9-day return window. Would you like me to connect you with our Application Manager if you have policy questions?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Today is the 9th day since my package arrived. Is it okay to submit a return request?",
    "output": "Hello! Yes, today is the final day of your 9-day return window, so you are eligible. Shall I go ahead and initiate that return request for you right now?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I received my product 3 days ago. What is the return deadline?",
    "output": "Hi there! Since your product arrived 3 days ago, you have 6 days remaining to request a return within our 9-day policy. Would you like me to help you start the return process?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Can I make a return 15 days after delivery if I pay for return shipping myself?",
    "output": "I appreciate your offer, but our 9-day return window is a fixed policy with no exceptions. Would you like me to escalate this ticket to a human supervisor for review?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I bought a shirt during the discount sale yesterday. Can I return it for a refund?",
    "output": "Hi there! I understand you'd like to return your shirt, but products purchased during an official discount sale are strictly final-sale and ineligible for returns. Would you like me to check your order details?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Can I exchange a discount sale item for a different size?",
    "output": "Hello! Unfortunately, discount sale items are final-sale and ineligible for exchanges of any kind. Shall I help you look up regular-priced catalog items instead?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Are discounted items covered under the standard return policy?",
    "output": "Hi! No, discounted sale items are exempt from returns and are final-sale purchases. Would you like me to review your order to confirm if it was part of the discount sale?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "If I haven't opened the package, can I return my discount sale purchase?",
    "output": "I understand the package is unopened, but our policy specifies that no returns are accepted for products bought at a discount sale. Would you like me to help you look up other eligible products?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Where is my same-day delivery? It has been 2 days since I placed the order.",
    "output": "Hi there! While we target fast fulfillment, we do not offer true same-day delivery as our shipments include a 3-day transit buffer. Could you please share your order number so I can check its tracking status?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My order is on day 3 of the delivery buffer and hasn't arrived. Is it late?",
    "output": "Hello! A 3-day transit duration falls fully within our standard delivery buffer framework. Shall I look up your tracking details to see where your package is right now?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Can I pay extra to expedite my order so it arrives on the same day?",
    "output": "Hello! We do not offer same-day delivery upgrades, as all orders are subject to our standard 3-day buffer framework. Would you like me to connect you with our Application Manager?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My package took 4 days to arrive, exceeding the buffer window. Can I get a shipping refund?",
    "output": "I apologize for the delay, as a 4-day wait exceeds our standard 3-day buffer framework. Could you please provide your order ID so I can escalate this delivery issue to a human supervisor?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "When is the next site-wide discount sale happening this year?",
    "output": "Hi! Our discount sale is held only once a calendar year on October 29 for Sree Narayana Guru Jayanti, with no other monthly sales. Would you like me to help you browse our regular catalog?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Do you hold a discount sale for Black Friday or New Year's?",
    "output": "Hello! We do not host seasonal markdowns, as our calendar features only one official discount sale held annually on October 29 for Sree Narayana Guru Jayanti. Shall I help you check out our standard items?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Why do you only have one discount sale a year instead of regular promotions?",
    "output": "I understand your curiosity about our promotional schedule; our policy is designed to host a single discount sale annually on October 29 for Sree Narayana Guru Jayanti. Would you like me to escalate this feedback to management?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Can I get a special coupon code since I missed the October 29 Sree Narayana Guru Jayanti sale?",
    "output": "I understand you're looking for a coupon code, but our support team operates under strict rules where our only sale happens once a year on October 29, with no manual coupon codes issued. Would you like me to help you browse regular pricing?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My blender stopped working after a week. Who should I email about this defect?",
    "output": "Hi there! I'm sorry your blender stopped working. For any product-related issues or defects, please email **`productqueris@myshpApp`** for direct assistance. Would you like me to help you find your order receipt?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Where should I send photos of a broken electronic item I received?",
    "output": "Hello! To share damage photos and report hardware faults, please email **`productqueris@myshpApp`** for all product-related issues. Shall I help you pull up your delivery date first?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "The seams on my jacket are torn. Who do I contact to report this manufacturing defect?",
    "output": "Hi! For product-related issues like torn seams, users can mail their concerns directly to **`productqueris@myshpApp`**. Would you like me to help format your message details?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Where do I direct questions about a product's material composition or care instructions?",
    "output": "Hello! For product-related queries regarding materials or care instructions, you can email **`productqueris@myshpApp`**. Would you like me to help you check the product details page first?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "The mobile app keeps crashing at checkout. Who should I contact to fix this bug?",
    "output": "Hi there! I'm sorry you're experiencing a checkout crash. For any app-related queries or technical issues, users can contact our Application Manager directly. Would you like me to help route you to them?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Where can I report a UI display glitch where text overlaps on the home screen?",
    "output": "Hello! For app-related queries regarding UI glitches, you can reach out directly to the Application Manager. Shall I help you draft a description of the bug?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I am locked out of my account due to a login error inside the app. Who handles this?",
    "output": "I understand you're locked out. For app-related queries and account access errors, users can contact the Application Manager. Would you like me to connect you with them?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "How can I provide feedback on the overall user interface design of the mobile app?",
    "output": "Hi! For app-related queries and design feedback, users are encouraged to contact our Application Manager. Would you like me to help structure your feedback points?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I bought a laptop for 45,000 and it stopped working after 6 weeks. Is it covered by warranty?",
    "output": "Hi there! Yes, for products priced above 5,000, you are covered by a two-month warranty. Since it has been 6 weeks, shall I help you email productqueris@myshpApp to claim your warranty service?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "What is the warranty period for a 3,000 item I purchased last week?",
    "output": "Hello! Our two-month warranty policy applies exclusively to products priced above 5,000, so items priced at 3,000 are not covered. Would you like me to check the product details page for other support options?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My 12,000 television broke down after 2 months and 2 weeks. Is it still under warranty?",
    "output": "I understand your television has a defect; however, since it has been over two months, it now falls outside our two-month warranty period for products above 5,000. Would you like me to escalate this ticket to a human supervisor?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "If I buy a 5,500 appliance, how long do I have to report manufacturer defects under warranty?",
    "output": "Hi! For any product priced above 5,000—such as your 5,500 appliance—you have a two-month warranty window to report defects. Would you like me to help you draft an email to productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I tried to place an order for 12,000 today, but checkout failed. Why won't it let me buy?",
    "output": "Hi there! Our app enforces a daily purchasing limit where a person can only buy up to 10,000 worth of products in a single day. Would you like me to help you save items in your cart for tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Can I make multiple separate orders today as long as each one is under the daily spending limit?",
    "output": "Hello! Our daily spending policy restricts total purchases to a maximum of 10,000 per person per calendar day across all orders. Shall I help you review your cart total?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "If I've already spent 7,000 today, can I place another order for 4,000 right now?",
    "output": "Since you have already spent 7,000 today, purchasing another 4,000 would exceed your remaining allowance under our 10,000 single-day limit. Would you like me to help you move some items to your cart for tomorrow?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "What is the maximum purchase amount allowed on the app per calendar day?",
    "output": "Hi! Each customer is permitted to purchase up to 10,000 worth of products in a single day. Would you like me to check your current order total to see if you are within this limit?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I want to return my fourth item this month, but the app won't let me submit the request. Why?",
    "output": "Hi there! Our policy limits customers to requesting a maximum of 3 product returns per calendar month. Would you like me to check how many returns you have already completed this month?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "How many total returns am I allowed to submit within a calendar month?",
    "output": "Hello! You are permitted to request up to 3 product returns per calendar month under our account policy. Shall I help you look up your recent return history?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "Does exchanging an item for a different size count toward my monthly limit of 3 returns?",
    "output": "Hi! Yes, all product returns and exchanges count toward your monthly allowance of 3 requests. Would you like me to check how many requests you have remaining?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "If I return 2 items today, how many return requests will I have left for the rest of the month?",
    "output": "Hello! If you complete 2 return requests today, you will have exactly 1 return request remaining for the rest of the calendar month. Shall I help you submit those requests now?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My glass vase arrived completely shattered in the box. What compensation do I get?",
    "output": "Hi there! I am so sorry your glass vase arrived damaged. Per our policy, when a person receives a damaged product, you will be refunded the complete bill amount along with a complementary 200 rupees coupon. Could you please email photos to productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "I received a broken appliance in my delivery today. Will I get my full money back?",
    "output": "Hello! Yes, for delivered products that arrive damaged, you will receive a complete refund of your bill amount plus a complimentary 200 rupees coupon. Shall I help you email productqueris@myshpApp?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "What bonus do I receive if my order arrives damaged due to transit issues?",
    "output": "Hi! If you are delivered a damaged product, our policy ensures you get a full refund of the complete bill amount plus a complimentary 200 rupees coupon. Would you like me to help you look up your order number?"
  },
  {
    "instruction": "Respond according to the support response policy for evaluation",
    "input": "My package arrived wet and ruined because of rain during transit. Does this count as a damaged delivery?",
    "output": "Hello! Yes, items ruined by weather or transit accidents qualify as delivered damaged products, meaning you will receive a complete bill refund and a complimentary 200 rupees coupon. Shall I help you email productqueris@myshpApp?"
  }
    ]

# Write out to local JSONL files
for filename, data in [("train.jsonl", train_samples), ("val.jsonl", val_samples), ("test.jsonl", test_samples)]:
    with open(filename, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print("JSONL datasets created successfully!")

JSONL datasets created successfully!


In [7]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 512
model_id = "Qwen/Qwen2.5-3B-Instruct"

print("Loading Qwen 2.5 3B model and tokenizer...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # Keeps memory low for Google Colab T4
)

# Attach LoRA adapters targeting all attention layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("Model successfully configured for QLoRA!")

Loading Qwen 2.5 3B model and tokenizer...
==((====))==  Unsloth 2026.9.12: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model successfully configured for QLoRA!


In [8]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# Load your generated jsonl files
dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "validation": "val.jsonl"}
)

# Standard prompt template enforcing tone, policy, and follow-up
support_prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = support_prompt_template.format(instruction=inst, input=inp, output=out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Map formatting across splits
formatted_dataset = dataset.map(formatting_func, batched=True)

# Configure the trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=10,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,
        output_dir="support_bot_output",
        report_to="none",
    ),
)

print("Starting training session...")
trainer.train()
print("Training completed successfully!")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

...

Starting training session...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 10 | Total steps = 190
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,Validation Loss
10,2.277535,2.225940
20,1.271985,1.587159
30,0.938606,1.402990
40,0.850066,1.267110
50,0.652160,1.240736
60,0.508543,1.271380
70,0.423040,1.224990
80,0.294434,1.329471
90,0.227238,1.371185
100,0.149879,1.451342


Unsloth: Restored added_tokens_decoder metadata in support_bot_output/checkpoint-190/tokenizer_config.json.


Training completed successfully!


In [10]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

inference_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Respond according to the support response policy: Maintain a warm, friendly tone, never argue with the customer, and provide a clear follow-up step.

### Input:
{input}

### Response:
"""

test_input = "When is the next site-wide discount sale happening this year?"
inputs = tokenizer([inference_prompt.format(input=test_input)], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True, temperature=0.1)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Respond according to the support response policy: Maintain a warm, friendly tone, never argue with the customer, and provide a clear follow-up step.

### Input:
When is the next site-wide discount sale happening this year?

### Response:
Hi there! Our single annual discount sale takes place once a year on October 29 for Sree Narayana Guru Jayanti. Would you like me to help you save items to your cart so you're ready when the sale begins?


In [22]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

inference_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Respond according to the support response policy: Maintain a warm, friendly tone, never argue with the customer, and provide a clear follow-up step.

### Input:
{input}

### Response:
"""

test_input = "My package arrived wet and ruined because of rain during transit. Does this count as a damaged delivery?"
inputs = tokenizer([inference_prompt.format(input=test_input)], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True, temperature=0.1)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Respond according to the support response policy: Maintain a warm, friendly tone, never argue with the customer, and provide a clear follow-up step.

### Input:
My package arrived wet and ruined because of rain during transit. Does this count as a damaged delivery?

### Response:
Hello! Yes, items ruined by weather or transit accidents qualify as delivered damaged products, meaning you will receive a complete bill refund and a complimentary 200 rupees coupon. Shall I help you email productqueris@myshpApp?
